# Institution Resolver v3 — Colab (uctan uca)

Bu notebook agir isleri (embedding uretimi, ES indeksleme, LLM hakem batch kosulari) yerel
Mac yerine Colab'in (ucretsiz GPU dahil) makinesinde calistirmak icin hazirlandi.

**Mimari (repo README'siyle ayni, sadece Docker yerine native process):**
```
raw CSV -> ingest/canonicalize -> embedding -> elastic (tek index)
sorgu: normalize -> elastic.search -> retrieve.signals -> gate (deterministik) -> judge (LLM) -> decide
```

**Onkosullar (bir kereligine):**
1. Runtime > Degistir > Donanim hizlandirici: **GPU (T4 yeterli)** sec.
2. Google Drive'inda `MyDrive/institution_resolver_v3/` altinda su klasorleri olustur ve doldur:
   - `data_raw/institution_parent.csv`, `data_raw/institution_subunit.csv` (ham veri, repo'da YOK — git'e commitlenmiyor)
   - `data_processed/` — **opsiyonel ama onerilir**: yerelde zaten uretilmis `parent_canonical.jsonl`,
     `subunit_canonical.jsonl`, `embeddings.npz`, `transform_report.json` dosyalarini buraya kopyala.
     `embeddings.npz` id+metin-hash'i eslesirse **23 dk'lik encode adimi tamamen atlanir** (bkz. `elastic/indexer.py::_compute_embeddings`).
   - `ollama_models/` — bos baslayabilir, ilk `ollama pull` sonrasi model burada kalici olur (~her seferinde yeniden inmez).
   - `output/` — batch sonuc CSV'lerinin yazilacagi yer.
   - `data_eval/` — batch/eval girdi CSV'lerini (orn. `benchmark_500_sample.csv`) buraya koy (bunlar da kisisel veri oldugu icin git'te yok).

Repo public oldugu icin klonlamak icin token gerekmiyor.

In [ ]:
!nvidia-smi

## 1) Drive baglama + kalici klasor yollari

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/institution_resolver_v3"
DRIVE_RAW = f"{DRIVE_ROOT}/data_raw"
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data_processed"
DRIVE_JOBS = f"{DRIVE_ROOT}/jobs"
DRIVE_EVAL = f"{DRIVE_ROOT}/data_eval"
DRIVE_OLLAMA = f"{DRIVE_ROOT}/ollama_models"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"

for p in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_JOBS, DRIVE_EVAL, DRIVE_OLLAMA, DRIVE_OUTPUT):
    os.makedirs(p, exist_ok=True)

print("raw:", os.listdir(DRIVE_RAW))
print("processed:", os.listdir(DRIVE_PROCESSED))
print("Drive klasorleri hazir:", DRIVE_ROOT)

Yukarida `raw` bos gorunuyorsa `institution_parent.csv` / `institution_subunit.csv`'yi Drive web
arayuzunden `data_raw/` altina yukle, sonra bu hucreyi tekrar calistir.

## 2) Repo (clone/pull) — public repo, token gerekmez

In [ ]:
REPO_DIR = "/content/institution_resolver_v3"
BRANCH = "feat/gate-asama1"  # guncel calisma dalin; ana dala gectiyseniz degistirin

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

## 3) Veri dizinlerini Drive'a sembolik-bagla

`data/raw`, `data/processed`, `data/jobs` gercek Drive klasorlerine link olur — boylece
embedding cache + islenmis JSONL'ler ve batch job ciktilari otomatik Drive'da kalicilasir,
elle kopyalamaya gerek kalmaz. `data/processed/transform_report.json` repo'da izlenen tek
dosya oldugu icin klon sonrasi gercek klasor olarak gelir; icerigini Drive'a tasiyip linke ceviriyoruz.

In [ ]:
import os, shutil

def _link(name: str, target: str) -> None:
    os.makedirs(target, exist_ok=True)
    link = f"data/{name}"
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for f in os.listdir(link):
            src, dst = f"{link}/{f}", f"{target}/{f}"
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs("data", exist_ok=True)
_link("raw", DRIVE_RAW)
_link("processed", DRIVE_PROCESSED)
_link("jobs", DRIVE_JOBS)

!ls -la data

## 4) Elasticsearch — native kurulum (Colab'da Docker daemon calismaz)

ES data dizini **kasitli olarak Drive'da degil**, Colab'in yerel diskinde tutulur:
Lucene segment dosyalari icin Drive'in FUSE baglantisi cok yavas/kararsizdir. Bunun yerine
her oturumda `data/processed` (Drive'daki cache) uzerinden hizlica yeniden indekslenir
— embedding cache sayesinde bu adim saniyeler surer, tekrar encode gerekmez.

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
EOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sudo -u esuser env ES_JAVA_OPTS="-Xms2g -Xmx2g" bash -c \
  'cd /content/es && nohup ./bin/elasticsearch > /content/es.log 2>&1 & echo $! > /content/es.pid'
echo "ES baslatildi, PID: $(cat /content/es.pid)"

In [ ]:
import time, requests

for _ in range(60):
    try:
        r = requests.get("http://localhost:9200/_cluster/health", timeout=2)
        if r.status_code == 200:
            print(r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ES 120s icinde ayaga kalkmadi, /content/es.log'a bak")

## 5) Ollama — native kurulum + model (Drive'da kalici)

`OLLAMA_MODELS` Drive'daki klasoru gosterir: model bir kez inince sonraki oturumlarda
tekrar indirilmez. Sunucu GPU varsa otomatik kullanir.

In [ ]:
%%bash
command -v ollama &> /dev/null || curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_MODELS"] = DRIVE_OLLAMA
subprocess.Popen(
    ["nohup", "ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
    env=os.environ,
)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama 60s icinde ayaga kalkmadi, /content/ollama.log'a bak")

!ollama pull gemma4:e4b

## 6) Python bagimliliklari (Colab'in mevcut CUDA'li torch'unu kullan — Dockerfile'daki CPU-only pin burada YOK)

In [ ]:
%%bash
cd /content/institution_resolver_v3
pip install -q -e ".[dev,embed,llm,api]"

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 7) Veri isleme + ES indeksleme

`data/processed` Drive'a bagli oldugu icin, orada zaten `embeddings.npz` varsa ve
id+metin-hash'i uyusuyorsa `--embeddings` adimi aninda biter (yeniden encode YOK).

In [ ]:
%%bash
cd /content/institution_resolver_v3
set -e
if [ ! -f data/processed/parent_canonical.jsonl ]; then
  echo "Islenmis veri yok, ham CSV'den uretiliyor..."
  inres3 build-data --raw-dir data/raw --out-dir data/processed
fi
inres3 setup-es
inres3 index --embeddings

## 8) Saglik kontrolu — tekli sorgu

In [ ]:
%%bash
cd /content/institution_resolver_v3
Q="gazi universitesi muhendislik fakultesi makine muhendisligi"
inres3 match "$Q"
echo "---"
inres3 gate "$Q"
echo "---"
inres3 judge "$Q"

## 9) Agir batch isi — girdi/cikti Drive uzerinden

Girdi CSV'ni `data_eval/` altina koy; `--resume` sayesinde oturum kesilirse kaldigi yerden devam eder.
Daha hafif alternatifler: `gate-batch` (LLM yok, hizli triyaj) veya `decide-batch` (hibrit).

In [ ]:
INPUT_CSV = f"{DRIVE_EVAL}/benchmark_500_sample.csv"
OUT_CSV = f"{DRIVE_OUTPUT}/batch_sonuc.csv"

!inres3 batch "{INPUT_CSV}" --out "{OUT_CSV}" --resume

## Oturum koptuyse / yeniden baslarken

Colab bosta ~90 dk veya ~12 saatte oturumu kapatir; arka plandaki ES/Ollama surecleri
bununla birlikte oler. Yeniden baglaninca su hucreleri sirayla tekrar calistirman yeterli:
**2 (repo) -> 3 (symlink) -> 4-5 (ES kur+baslat+bekle) -> 6-7 (Ollama kur+baslat+pull) -> 8 (pip install)
-> 9 (index)**. `embeddings.npz` ve Ollama modeli Drive'da kalici oldugu icin bu adimlar
ilk seferkinden cok daha hizli biter (yeniden encode/indirme yok).